In [2]:
from pathlib import Path
import pandas as pd

csv_path = Path("../data/Medicaldataset.csv")
if not csv_path.exists():
    csv_path = Path("data/Medicaldataset.csv")

df = pd.read_csv(csv_path)
df.head()

,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin,Result
0,64,1,66,160,83,160.0,1.80,0.012,negative
1,21,1,94,98,46,296.0,6.75,1.060,positive
2,55,1,64,160,77,270.0,1.99,0.003,negative
3,64,1,70,120,55,270.0,13.87,0.122,positive
4,55,1,64,112,65,300.0,1.08,0.003,negative


In [3]:
df_clean = df.copy()
df_clean.columns = df_clean.columns.str.strip()

for column in df_clean.select_dtypes(include="object").columns:
    df_clean[column] = df_clean[column].str.strip()

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print(f"Rows after cleaning: {len(df_clean)}")
print("Missing values per column:")
print(df_clean.isna().sum())
df_clean.head()

Rows after cleaning: 1319
Missing values per column:
Age                         0
Gender                      0
Heart rate                  0
Systolic blood pressure     0
Diastolic blood pressure    0
Blood sugar                 0
CK-MB                       0
Troponin                    0
Result                      0
dtype: int64


,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin,Result
0,64,1,66,160,83,160.0,1.80,0.012,negative
1,21,1,94,98,46,296.0,6.75,1.060,positive
2,55,1,64,160,77,270.0,1.99,0.003,negative
3,64,1,70,120,55,270.0,13.87,0.122,positive
4,55,1,64,112,65,300.0,1.08,0.003,negative


In [4]:
df_features = df_clean.copy()

systolic_pressure = df_features["Systolic blood pressure"]
diastolic_pressure = df_features["Diastolic blood pressure"]

# Derived features from existing vital-sign measurements.
df_features["Pulse Pressure"] = systolic_pressure - diastolic_pressure
df_features["Mean Arterial Pressure"] = diastolic_pressure + df_features["Pulse Pressure"] / 3
df_features["Shock Index"] = df_features["Heart rate"].div(
    systolic_pressure.where(systolic_pressure != 0)
)

df_features.head()

,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin,Result,Pulse Pressure,Mean Arterial Pressure,Shock Index
0,64,1,66,160,83,160.0,1.80,0.012,negative,77,108.666667,0.412500
1,21,1,94,98,46,296.0,6.75,1.060,positive,52,63.333333,0.959184
2,55,1,64,160,77,270.0,1.99,0.003,negative,83,104.666667,0.400000
3,64,1,70,120,55,270.0,13.87,0.122,positive,65,76.666667,0.583333
4,55,1,64,112,65,300.0,1.08,0.003,negative,47,80.666667,0.571429


In [5]:
from sklearn.model_selection import train_test_split

X = df_features.drop(columns=["Result"])
y = df_features["Result"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training set:   {X_train.shape[0]} rows")
print(f"Validation set: {X_validation.shape[0]} rows")
print(f"Test set:       {X_test.shape[0]} rows")

ModuleNotFoundError: No module named 'sklearn'